# Routing Pattern
### Sending work to the right specialist

Three signals your process needs routing:
1. **Ambiguous intent** — work arrives in natural language
2. **Multiple specialists** — different agents with distinct capabilities
3. **Cost of misroute** — wrong handler is more expensive than classification

### Router Types

| Type | Best For | Trade-off |
|---|---|---|
| **Rule-based** | Stable, finite categories | Unmatched inputs become exceptions |
| **LLM classification** | Broad, varied input | Higher latency per decision |
| **Fine-tuned classifier** | High-volume, well-defined domains | Requires training data |
| **Hybrid** | Production — speed + accuracy | More complex to build |

### Business Use Case: AnyComp Telecom Customer Service Routing

Customer inquiries arrive as free-text. We use **LLM classification** because the input is ambiguous with wide vocabulary variation.

### Architecture

```
Customer Message → Router Agent (LLM classifier)
                        │
          ┌───────┬─────┼───────┬─────────┐
          ▼       ▼     ▼       ▼         ▼
       Billing  Tech   Plan   General  Retention
       Agent    Agent  Agent  Agent    Agent
                        │
               Specialist response
```



## Install Dependencies

Run this cell first, then **restart the kernel** and continue.

In [ ]:
!pip install -q strands-agents>=1.36.0 strands-agents-tools boto3 "typing_extensions>=4.12.0"

## Install Dependencies

**Restart the kernel** after installing, then continue.

## Setup: Specialist Agents and Test Queries

In [ ]:
import json

REGION = 'us-east-1'

# Test queries spanning different domains
TEST_QUERIES = [
    "I was charged twice on my last bill for international roaming. Can you fix this?",
    "My internet has been dropping every evening for the past week. Speed test shows 5 Mbps instead of 100.",
    "I want to upgrade from Basic to Premium. What are the features and how much more will it cost?",
    "What are your store hours? I need to pick up a new SIM card.",
    "I want to cancel my service. Your competitor is offering a better deal.",
]

CATEGORIES = ['billing', 'technical_support', 'plan_change', 'general_inquiry', 'retention']

print(f'{len(TEST_QUERIES)} test queries across {len(CATEGORIES)} categories')

## Implementation 1: Strands Agents

The router is a Strands `Agent` whose tools are the specialist agents. The router’s system prompt tells it to classify the intent and call the right specialist. The LLM handles both classification and dispatch in one reasoning step.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    region_name=REGION,
    max_tokens=4096,
)

# ── Specialist agents ─────────────────────────────────────────────────────────

@tool
def handle_billing(query: str) -> str:
    """Handle billing disputes, payment issues, and charge corrections.
    Args:
        query: The customer's billing-related query
    """
    agent = Agent(model=model, system_prompt=(
        "You are a billing specialist at AnyComp Telecom. "
        "Resolve billing disputes, explain charges, and process corrections. "
        "Be specific about amounts and dates."
    ))
    return str(agent(query))

@tool
def handle_technical_support(query: str) -> str:
    """Handle network issues, connectivity problems, and speed complaints.
    Args:
        query: The customer's technical issue description
    """
    agent = Agent(model=model, system_prompt=(
        "You are a technical support engineer at AnyComp Telecom. "
        "Diagnose network issues, suggest troubleshooting steps, and escalate if needed."
    ))
    return str(agent(query))

@tool
def handle_plan_change(query: str) -> str:
    """Handle plan upgrades, downgrades, and feature inquiries.
    Args:
        query: The customer's plan change request
    """
    agent = Agent(model=model, system_prompt=(
        "You are a plan specialist at AnyComp Telecom. "
        "Help customers compare plans, calculate costs, and process changes."
    ))
    return str(agent(query))

@tool
def handle_general_inquiry(query: str) -> str:
    """Handle general questions about store hours, SIM cards, and other non-specific topics.
    Args:
        query: The customer's general question
    """
    agent = Agent(model=model, system_prompt=(
        "You are a general customer service agent at AnyComp Telecom. "
        "Answer general questions about stores, services, and policies."
    ))
    return str(agent(query))

@tool
def handle_retention(query: str) -> str:
    """Handle cancellation requests and retention offers.
    Args:
        query: The customer's cancellation or retention-related message
    """
    agent = Agent(model=model, system_prompt=(
        "You are a retention specialist at AnyComp Telecom. "
        "Empathize with the customer, understand their reason for leaving, "
        "and offer a compelling reason to stay."
    ))
    return str(agent(query))


# ── Router agent ──────────────────────────────────────────────────────────────
router = Agent(
    model=model,
    system_prompt=(
        "You are the customer service router for AnyComp Telecom. "
        "Classify each customer message and route it to the right specialist:\n"
        "- handle_billing: billing disputes, charges, payments\n"
        "- handle_technical_support: network issues, speed, connectivity\n"
        "- handle_plan_change: upgrades, downgrades, plan comparisons\n"
        "- handle_general_inquiry: store hours, SIM cards, general questions\n"
        "- handle_retention: cancellation requests, competitor mentions\n\n"
        "Call exactly ONE specialist per query. Before calling, state which category you classified it as."
    ),
    tools=[handle_billing, handle_technical_support, handle_plan_change, handle_general_inquiry, handle_retention],
)

print("=" * 60)
print("STRANDS — Routing Agent")
print("=" * 60)
for q in TEST_QUERIES:
    print(f"\nQuery: {q}")
    print(f"Response: {router(q)}")
    print("-" * 40)

## Why LLM Classification for This Use Case

We chose **LLM classification** over rule-based or fine-tuned approaches because:

- Customer messages are **free-text with wide vocabulary variation** — "my bill is wrong", "overcharged", "charged twice" all mean the same thing
- The categories are **broad and overlapping** — "I want to cancel because my bill is too high" touches both retention and billing
- We don’t have **labeled training data** for a fine-tuned classifier
- The **cost of misrouting** is moderate (customer gets transferred) not catastrophic (compliance failure)

In production, you’d monitor classification accuracy and consider a **hybrid approach**: a fast classifier for common patterns with LLM fallback for ambiguous cases.